# Notebook 3: The SGET Architecture

This notebook walks through how SGET is structured so you can read, modify, and extend the codebase. We'll cover the central model, the signal-based communication pattern, the layer configuration system, the 2D layout, and the GUI widget tree.

## Prerequisites
- Notebooks 1 and 2 completed
- Neo4j running with the example DSG loaded (Notebook 2 does this)

## 1. Architecture Overview

SGET follows a **model-signal-view** pattern (Qt's version of MVC):

```
 JSON file ──► spark_dsg ──► heracles bulk load ──► Neo4j
                                                      │
                                              SceneGraphModel
                                           (cache + Qt signals)
                                          /        |          \
                                    GraphView  LayerPanel  PropertyPanel
                                    (2D view)  (toggles)   (editor)
```

**Key principle**: all data flows through the `SceneGraphModel`. Views never talk to Neo4j directly — they read from the model's in-memory cache and react to Qt signals when data changes.

**Why a cache?** Neo4j queries are fast, but not fast enough for rendering hundreds of nodes at 60fps. The model maintains a dict of node properties and a list of edges, populated from Neo4j on load. Mutations update both the cache and Neo4j, then emit signals.

**Why not keep spark_dsg in memory?** spark_dsg's pybind11 bindings are designed for bulk construction and serialization, not fine-grained mutation with change notification. The DSG object is transient — used only at the load/save boundaries.

## 2. The SceneGraphModel

The `SceneGraphModel` (a `QObject`) is the heart of the application. Let's see it in action — we'll create one, connect it to Neo4j, load a graph, and observe how signals fire.

In [ ]:
# Qt requires a QApplication instance for signals to work, even without a GUI.
import os

from heracles.utils import get_labelspace
from PySide6.QtWidgets import QApplication

from sget.backend.scene_graph_model import SceneGraphModel

app = QApplication.instance() or QApplication([])

model = SceneGraphModel()
model.connect("neo4j://127.0.0.1:7687", "neo4j", "neo4j_pw")

# Set labelspaces (same as app.py does from CLI args).
object_labels = get_labelspace("ade20k_mit_label_space.yaml")
room_labels = get_labelspace("b45_label_space.yaml")
model.set_labelspaces(
    object_labels={v: k for k, v in object_labels.items()},
    room_labels={v: k for k, v in room_labels.items()},
)

print(f"Connected: {model.connected}")

In [ ]:
# SignalSpy — a simple helper that records every emission of a Qt signal.
# This is the same pattern used in our test suite (tests/test_scene_graph_model.py).
class SignalSpy:
    def __init__(self, signal):
        self.calls = []
        signal.connect(lambda *args: self.calls.append(args))


# Watch the graph_loaded signal.
loaded_spy = SignalSpy(model.graph_loaded)

# Load the example DSG.
DSG_PATH = os.path.expanduser(
    "~/software/mit/sget/heracles/heracles/examples/scene_graphs/example_dsg.json"
)
model.load_from_json(DSG_PATH)

print(f"graph_loaded fired: {len(loaded_spy.calls)} time(s)")
print(f"Cache: {len(model.get_all_nodes())} nodes, {len(model.get_edges())} edges")

In [ ]:
# Now watch what happens when we add a node — the model updates the cache,
# writes to Neo4j, and emits node_added.
from heracles import constants

add_spy = SignalSpy(model.node_added)

model.add_node(
    constants.OBJECTS,
    "o(99)",
    {
        "pos_x": 0.0,
        "pos_y": 0.0,
        "pos_z": 0.0,
        "bbox_x": 0.0,
        "bbox_y": 0.0,
        "bbox_z": 0.0,
        "bbox_l": 1.0,
        "bbox_w": 1.0,
        "bbox_h": 1.0,
        "class": "box",
        "name": "demo_node",
    },
)

print(f"node_added fired: {add_spy.calls}")
print(f"Node in cache: {model.get_node('o(99)') is not None}")
print(f"Node layer:    {model.get_node_layer('o(99)')}")

# Clean up.
model.remove_node("o(99)")

## 3. Layer Configuration — Single Source of Truth

SGET uses `utils/colors.py` as the **single source of truth** for layer configuration. It defines `LAYER_STYLES` — an ordered list of `LayerStyle` dataclasses that specify each layer's label, ID, display name, color, and category character.

Everything else derives from this list: the model's `LAYER_ORDER`, the layer panel's rows, the graph view's node colors, and the 2D layout's band positions.

In [ ]:
from sget.utils.colors import LAYER_STYLES, STYLE_BY_LABEL

# LAYER_STYLES defines the hierarchy order (top to bottom in the UI).
for style in LAYER_STYLES:
    print(
        f"  {style.display_name:12s}  layer_id={style.layer_id:2d}  "
        f"label={style.layer_label:10s}  color={style.color}  char={style.category_char}"
    )

# Quick lookup by heracles label string.
print(f"\nRoom color: {STYLE_BY_LABEL['Room'].color}")

## 4. The 2D Layout

The graph view arranges nodes in **horizontal bands** — one per layer — with Buildings at the top and Objects at the bottom. Within each band, nodes are positioned using NetworkX's spring layout, which clusters connected siblings together.

The layout operates on the model's cache (plain dicts), not on spark_dsg objects. This keeps it simple and fast.

In [ ]:
from sget.utils.layout import compute_layout

# The layout function takes the same data structures the model cache provides.
nodes = model.get_all_nodes()
node_layers = {ns: model.get_node_layer(ns) for ns in nodes}
edges = model.get_edges()

positions = compute_layout(nodes, node_layers, edges)

# Show a few positions and how they're grouped by layer band.
print(f"Computed positions for {len(positions)} nodes\n")
print("Sample positions by layer:")
y_by_layer = {}
for ns, (x, y) in list(positions.items())[:200]:
    layer = node_layers[ns]
    y_by_layer.setdefault(layer, y)

for layer, y in sorted(y_by_layer.items(), key=lambda kv: kv[1]):
    count = sum(1 for ns in positions if node_layers.get(ns) == layer)
    print(f"  {layer:10s}: y={y:6.0f}  ({count} nodes)")

## 5. The GUI Widget Tree

The GUI is built with **PySide6** (Qt for Python). Here's the widget hierarchy:

```
MainWindow (QMainWindow)
├── Central Widget: GraphView (QWidget)
│   └── _ZoomableGraphicsView (QGraphicsView)
│       └── QGraphicsScene
│           ├── NodeItem (QGraphicsEllipseItem) × N  [selectable]
│           └── EdgeItem (QGraphicsLineItem) × M     [selectable]
├── Left Dock: LayerPanel (QWidget)
│   └── Per-layer rows: [checkbox] [swatch] [label] [count]
├── Right Dock: PropertyPanel (QWidget)
│   └── QFormLayout with editable fields + Apply button
└── Menu Bar
    ├── File: Open, Save As, Connect to Neo4j, Quit
    ├── Edit: Add Node (Ctrl+N), Delete Selected (Del), Group (Ctrl+G)
    └── View: toggle Layers/Properties docks
```

### Dialogs (modal, opened from menu or context menu)
- **AddNodeDialog** (`widgets/add_node_dialog.py`): pick layer, position, name, class → auto-generates nodeSymbol
- **GroupDialog** (`widgets/group_dialog.py`): select nodes in same layer → create parent in higher layer with CONTAINS edges
- **ConnectionDialog** (`widgets/connection_dialog.py`): enter Neo4j URI/credentials at runtime

### Signal flow for node selection
1. User clicks a `NodeItem` in the `QGraphicsScene`
2. `QGraphicsScene.selectionChanged` fires
3. `GraphView._on_scene_selection_changed()` reads the selected items
4. Calls `model.set_selection([node_symbols...])`
5. Model emits `selection_changed` signal
6. `PropertyPanel._on_selection_changed()` populates the form
7. `GraphView._on_selection_changed()` updates gold highlights (with a guard to prevent recursion)

### Incremental view updates
When a node or edge is added/removed through the model, the graph view doesn't recompute the entire layout. Instead:
- `model.node_added` → `GraphView._on_node_added()` creates a single `NodeItem` at the layer's centroid
- `model.node_removed` → `GraphView._on_node_removed()` removes the `NodeItem` and its connected `EdgeItem`s
- `model.edge_added` / `edge_removed` → add/remove a single `EdgeItem`

Full layout recomputation only happens on `graph_loaded` (File → Open).

### Context menu (right-click in graph view)
- **2 nodes selected** → "Add Edge"
- **Nodes selected** → "Delete N Node(s)"
- **Edges selected** → "Delete N Edge(s)"

## 6. Extending SGET

Here are practical pointers for common modifications:

**Adding a new property to the property panel**:
Edit `views/property_panel.py` → `_show_node()`. Add a widget for your property (QLineEdit, QSpinBox, etc.), store it in `self._widgets`, and handle it in `_on_apply()`. The key mapping `"pos" → "center"` is where widget names translate to Neo4j property names.

**Adding a new menu action**:
Edit `main_window.py` → `_setup_menus()`. Add a `QAction` to the appropriate menu and connect it to a handler method. See `_add_node` and `_group_selected` for examples of dialog-based actions.

**Adding a new dialog**:
Follow the pattern in `widgets/add_node_dialog.py` or `widgets/group_dialog.py`:
1. Subclass `QDialog` with form fields
2. Accept/reject buttons via `QDialogButtonBox`
3. A `get_result()` or `execute_*()` method that returns data on accept, None on cancel
4. Call from `main_window.py` handler

**Adding a new layer**:
1. Add a `LayerStyle` entry to `LAYER_STYLES` in `utils/colors.py`
2. Add a CREATE template function in `neo4j_crud.py` and register it in `_CREATE_DISPATCH`
3. Add the layer to `_PARENT_LAYERS` in `widgets/group_dialog.py` if it participates in grouping
4. Everything else (model, layout, layer panel, add node dialog) picks it up automatically

**Adding a new view or panel**:
1. Create a new widget class that takes the `SceneGraphModel` in its constructor
2. Connect to the model's signals (`graph_loaded`, `selection_changed`, `node_added`, etc.)
3. Add it as a dock widget in `main_window.py`

**NodeSymbol generation**:
When adding nodes programmatically, use `_next_node_symbol(model, layer_label)` from `widgets/add_node_dialog.py`. It scans existing nodes to find the next available index for the layer's category character.

**Running the test suite**:
```bash
pytest                                  # All 47 tests (requires Neo4j)
pytest tests/test_neo4j_crud.py         # CRUD layer (23 tests)
pytest tests/test_scene_graph_model.py  # Model tests (24 tests)
pytest -k "selection"                   # Tests matching a keyword
```

In [ ]:
# Clean up.
model.disconnect()
print("Done! You're ready to start contributing to SGET.")

## Summary

You've learned:
- SGET uses a **model-signal-view** pattern: `SceneGraphModel` is the single source of truth
- The model maintains a **dual store** (Neo4j + in-memory cache) and emits **Qt signals** on every mutation
- `utils/colors.py` is the **single source of truth** for layer configuration — everything derives from `LAYER_STYLES`
- The 2D layout uses **hierarchical bands** with per-band spring layout
- The GUI is a `QMainWindow` with dockable panels, an Edit menu (Add Node, Delete, Group), and a right-click context menu
- **Incremental updates**: node/edge add/remove signals update individual QGraphicsItems without full layout recompute
- **Dialogs**: AddNodeDialog, GroupDialog, ConnectionDialog follow a consistent pattern (QDialog + get_result/execute)
- To extend SGET: add to `LAYER_STYLES` for new layers, connect to model signals for new views, follow the dialog pattern for new operations

For the full implementation details, see the source files directly — they are thoroughly documented with design-choice comments.